# Proyecto 2 - Modelo de Clasificación Climática

## 03_Clasificacion_Modelos.ipynb

Este notebook entrena un modelo de clasificación para predecir categorías de temperatura (`cold`, `mild`, `hot`) usando las características extraídas.

### Objetivos
- Cargar los datos con características extraídas
- Definir un objetivo de clasificación
- Entrenar un clasificador de bosques aleatorios
- Evaluar desempeño con métricas y matriz de confusión

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import seaborn as sns
import matplotlib.pyplot as plt

sns.set(style='whitegrid', font_scale=1.1)

train_path = '../data/train_feature_engineered.csv'
train_df = pd.read_csv(train_path)

print('Shape:', train_df.shape)
train_df.head()

## Preparar objetivo y características

In [ ]:
feature_columns = [
    'year', 'month', 'day', 'dayofweek', 'quarter', 'is_weekend',
    'humidity', 'wind_speed', 'meanpressure', 'humidity_pressure_ratio',
    'meantemp_rolling_3', 'meantemp_rolling_7',
    'humidity_rolling_3', 'humidity_rolling_7',
    'wind_speed_rolling_3', 'wind_speed_rolling_7',
    'meanpressure_rolling_3', 'meanpressure_rolling_7'
]

X = train_df[feature_columns]
y = train_df['temp_category']

encoder = LabelEncoder()
y_encoded = encoder.fit_transform(y)

X_train, X_val, y_train, y_val = train_test_split(X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded)

print('Train:', X_train.shape, 'Validation:', X_val.shape)

## Entrenar Random Forest

In [ ]:
model = RandomForestClassifier(n_estimators=150, random_state=42)
model.fit(X_train, y_train)

train_pred = model.predict(X_train)
val_pred = model.predict(X_val)

print('Accuracy Train:', accuracy_score(y_train, train_pred))
print('Accuracy Validation:', accuracy_score(y_val, val_pred))

## Evaluación del modelo

In [ ]:
print('Classification Report Validation:')
print(classification_report(y_val, val_pred, target_names=encoder.classes_))

cm = confusion_matrix(y_val, val_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=encoder.classes_, yticklabels=encoder.classes_)
plt.title('Matriz de confusión - Validación')
plt.xlabel('Predicho')
plt.ylabel('Real')
plt.show()

## Importancia de las características

In [ ]:
importances = pd.Series(model.feature_importances_, index=feature_columns).sort_values(ascending=False)
plt.figure(figsize=(10, 6))
importances.head(12).plot(kind='bar', color='teal')
plt.title('Top 12 características más importantes')
plt.ylabel('Importancia')
plt.show()

## Conclusiones

- El modelo de clasificación permite distinguir rápidamente tres estados de temperatura.
- La extracción de características temporales y las medias móviles aportan señal útil.
- El resultado puede servir como análisis complementario previo a modelos de predicción más avanzados.